# Lesson 5.6 — Preserving Provenance & Handling Uncertainty in Multi-Source Synthesis

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Preserve claim-source mappings through summarization and synthesis
- Annotate conflicting statistics with source attribution instead of picking one
- Require publication/collection dates to avoid misreading temporal differences
- Render different content types appropriately instead of one uniform format

**You will produce:** A claim-source schema, a conflict-annotation merge, and a content-type-aware renderer.

## Concepts

Source attribution is lost when findings are **compressed without preserving claim-source mappings**. Build reliability into synthesis:

- Require subagents to output **structured claim-source mappings** (source URLs, document names, relevant excerpts) that downstream agents **preserve and merge**.
- For **conflicting statistics** from credible sources, **annotate the conflict with source attribution** rather than arbitrarily selecting one value; let the coordinator reconcile.
- Structure reports to **distinguish well-established findings from contested ones**, keeping original source characterizations and methodological context.
- Require **publication/collection dates** so temporal differences aren't misread as contradictions.
- **Render content types appropriately** - financial data as tables, news as prose, technical findings as structured lists - rather than flattening everything to one format.

> **Reference:** [Anthropic - citations & grounded synthesis](https://docs.anthropic.com/en/docs/build-with-claude/citations)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — A claim-source mapping schema

Every claim carries its source, excerpt, and date through synthesis.

In [ ]:
def claim(text, source, excerpt, published):
    return {"claim": text, "source": source, "excerpt": excerpt, "published": published}

findings = [
    claim("Market grew 12%", "https://a.com/report", "...grew 12% YoY...", "2026-02"),
    claim("Market grew 18%", "https://b.com/study", "...up 18%...", "2025-09"),
]
import json; print(json.dumps(findings, indent=2))

## Code-Along: Step 2 — Annotate conflicts instead of choosing

Conflicting numbers are kept with attribution and dates - not silently resolved.

In [ ]:
def merge_with_conflicts(findings, topic):
    values = [(f["claim"], f["source"], f["published"]) for f in findings]
    conflict = len({v[0] for v in values}) > 1
    return {"topic": topic, "conflict": conflict,
            "values": [{"value": c, "source": s, "published": p} for c, s, p in values]}

print(json.dumps(merge_with_conflicts(findings, "market growth"), indent=2))

## Code-Along: Step 3 — Content-type-aware rendering

Financial -> table, news -> prose, technical -> list.

In [ ]:
def render(content_type, data):
    if content_type == "financial":
        rows = "\n".join(f"| {k} | {v} |" for k, v in data.items())
        return "| metric | value |\n|---|---|\n" + rows
    if content_type == "news":
        return " ".join(data)
    if content_type == "technical":
        return "\n".join(f"- {d}" for d in data)
    return str(data)

print(render("financial", {"revenue": "$12M", "growth": "12%"}))
print(render("technical", ["uses async IO", "retries 3x"]))

## Exercise

Add a `report_sections(findings)` that splits output into **Well-Established** (single consistent value) vs **Contested** (conflicting values with attribution) sections. Then assert that two findings differing only by **publication date** are NOT flagged as a contradiction when their values agree - proving temporal handling.

In [ ]:
# TODO: implement report_sections(findings) -> {'well_established': [...], 'contested': [...]}
# TODO: assert same-value/different-date findings are not marked contested

## Key Takeaways

- Preserve claim-source mappings (URL, excerpt, date) through summarization and synthesis.
- Annotate conflicting statistics with attribution instead of arbitrarily choosing one.
- Require publication/collection dates so temporal gaps aren't read as contradictions.
- Render financial as tables, news as prose, technical as lists - don't flatten everything.

---

**Phase complete!** Return to the [study plan README](../README.md) to continue.